# CardioIA — Fase 4 (Parte 2): classificação ECG com CNN

Compara **CNN simples** (do zero) e **transfer learning** (VGG16). Métricas: acurácia, matriz de confusão, precision, recall e F1.

**Aviso:** protótipo acadêmico; não substitui avaliação médica.

In [ ]:
# %pip install tensorflow pillow scikit-learn matplotlib seaborn

import sys
from pathlib import Path
import numpy as np

FASE4 = Path('..').resolve()
sys.path.insert(0, str(FASE4))

from src.preprocessamento import load_splits, preprocess_batch
from src.modelos import build_cnn_simples, build_transfer_learning
from src.avaliacao import compute_metrics, plot_confusion_matrix, plot_training_history, plot_metrics_comparison
from tensorflow import keras

In [ ]:
meta = load_splits()
classes = meta['classes']

def load_xy(split):
    paths = [Path(p) for p in split['paths']]
    X = preprocess_batch(paths)
    y = np.array(split['y'], dtype=np.int32)
    return X, y

X_train, y_train = load_xy(meta['splits']['train'])
X_val, y_val = load_xy(meta['splits']['val'])
X_test, y_test = load_xy(meta['splits']['test'])
print(f'Treino {X_train.shape} | Val {X_val.shape} | Teste {X_test.shape}')

## CNN simples (do zero)

In [ ]:
MODELS = FASE4 / 'models'
IMAGES = FASE4 / 'docs' / 'imagens'
MODELS.mkdir(exist_ok=True)
IMAGES.mkdir(exist_ok=True)

cnn = build_cnn_simples(num_classes=len(classes))
cnn.summary()

cb = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(str(MODELS / 'ecg_cnn_simples_best.keras'), monitor='val_accuracy', save_best_only=True),
]
hist_cnn = cnn.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=8, batch_size=16, callbacks=cb, verbose=1)

y_pred = np.argmax(cnn.predict(X_test, verbose=0), axis=1)
metrics_cnn = compute_metrics(y_test, y_pred, classes)
print(metrics_cnn['classification_report'])
plot_training_history(hist_cnn, IMAGES / 'historico_cnn_simples.png', 'CNN simples')
plot_confusion_matrix(y_test, y_pred, classes, 'CNN simples', IMAGES / 'matriz_confusao_cnn_simples.png')

## Transfer Learning (VGG16)

In [ ]:
transfer = build_transfer_learning(num_classes=len(classes), backbone='vgg16')
transfer.summary()

cb_tl = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(str(MODELS / 'ecg_transfer_best.keras'), monitor='val_accuracy', save_best_only=True),
]
hist_tl = transfer.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=6, batch_size=16, callbacks=cb_tl, verbose=1)

y_pred_tl = np.argmax(transfer.predict(X_test, verbose=0), axis=1)
metrics_tl = compute_metrics(y_test, y_pred_tl, classes)
print(metrics_tl['classification_report'])
plot_training_history(hist_tl, IMAGES / 'historico_transfer_learning.png', 'Transfer Learning VGG16')
plot_confusion_matrix(y_test, y_pred_tl, classes, 'Transfer Learning VGG16', IMAGES / 'matriz_confusao_transfer_learning.png')
plot_metrics_comparison({'CNN simples': metrics_cnn, 'Transfer Learning': metrics_tl}, IMAGES / 'metricas_comparacao.png')

## Inferência (mesma API do Flask)

In [ ]:
from src.inferencia import predict_image

sample = Path(meta['splits']['test']['paths'][0])
resultado = predict_image(sample, MODELS / 'ecg_transfer_best.keras')
print(f'Imagem: {sample.name}')
print(resultado)